# MapTR official nuScenes baseline

This notebook runs the unmodified official nuScenes MapTR dataset path.


## Data flow

この notebook の入力、途中データ、出力、処理順は [MapTR README](../README.md#01-nuscenes-baseline) にまとめています。

ここでは README の flow に沿って、公式 nuScenes sample で次を実行します。

- 公式変換済み pkl の入力確認
- `types.py` 形式との contract 比較
- 公式 pipeline での推論
- GT と prediction の BEV 可視化


### Output check: paths

この cell では、notebook が参照する file path が正しいかを見ます。

OK の目安:
- `config`, `checkpoint`, `nuscenes_root`, `train_info`, `val_info`, `selected_ann` がすべて `exists=True`
- `SPLIT` を変えた場合、`selected_ann` が意図した train/val pkl になっている

ここで `exists=False` が出たら、以降の dataset build や推論を見る前に path を直します。


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import os
import sys

WORKSPACE_ROOT = Path('/home/apollo-22/hdmap-driving-workspace')
HDMAP_MODEL_BENCH_ROOT = WORKSPACE_ROOT / 'repos' / 'hdmap-model-bench'
MAPTR_MODEL_DIR = HDMAP_MODEL_BENCH_ROOT / 'models' / 'maptr'
MAPTR_ROOT = MAPTR_MODEL_DIR / 'external' / 'MapTR'

CONFIG_PATH = MAPTR_ROOT / 'projects' / 'configs' / 'maptr' / 'maptr_tiny_r50_24e_bevformer.py'
CHECKPOINT_PATH = WORKSPACE_ROOT / 'checkpoints' / 'maptr' / 'official' / 'maptr_tiny_r50_24e_bevformer.pth'
NUSCENES_ROOT = WORKSPACE_ROOT / 'dataset' / 'nuscenes' / 'nuscenes'
TRAIN_INFO_FILE = NUSCENES_ROOT / 'nuscenes_map_infos_temporal_train.pkl'
VAL_INFO_FILE = NUSCENES_ROOT / 'nuscenes_map_infos_temporal_val.pkl'

SPLIT = 'val'
ANN_FILE = {'train': TRAIN_INFO_FILE, 'val': VAL_INFO_FILE}[SPLIT]

SAMPLE_INDEX = 0
SCORE_THRESHOLD = 0.30
DEVICE = 'cuda:0'

for path in [HDMAP_MODEL_BENCH_ROOT, MAPTR_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

for name, check_path in {
    'config': CONFIG_PATH,
    'checkpoint': CHECKPOINT_PATH,
    'nuscenes_root': NUSCENES_ROOT,
    'train_info': TRAIN_INFO_FILE,
    'val_info': VAL_INFO_FILE,
    'selected_ann': ANN_FILE,
}.items():
    print(f'{name:14s}: {check_path} exists={check_path.exists()}')


### Output check: runtime environment

この cell では、kernel が MapTR 実行環境になっているかを見ます。

OK の目安:
- missing module の error が出ない
- `cuda available: True`
- `python:` が `maptr` conda env を指している

ここで CUDA が false の場合、dataset inspection はできても MapTR 推論は進めません。


In [ ]:
required = ['torch', 'mmcv', 'mmdet', 'mmdet3d', 'nuscenes', 'pyquaternion', 'cv2', 'matplotlib', 'numpy']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'Missing modules in this kernel: {missing}. Activate the MapTR environment.')

import cv2
import mmcv
import numpy as np
import torch
import matplotlib.pyplot as plt
from pyquaternion import Quaternion

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not DEVICE.startswith('cuda') or not torch.cuda.is_available():
    raise RuntimeError('Run this notebook with a GPU-enabled MapTR kernel.')

## Build official MapTR dataset: `ANN_FILE` -> `dataset`

公式 converter が生成した `nuscenes_map_infos_temporal_{train,val}.pkl` を `ann_file` として読み込みます。この pkl が MapTR 用の正規データで、sensor path、calibration、can_bus、pose、timestamp、local vector map annotation の基準になります。

この notebook では 1 sample の入力 contract を確認するため、`map_ann_file` は使いません。`map_ann_file` は MapTR dataset の評価用 GT JSON を保存・再利用するときの出力先です。


### Output check: official dataset build

この cell では、公式変換済み pkl を MapTR dataset として読めるかを見ます。

OK の目安:
- dataset type が `CustomNuScenesLocalMapDataset` 系になっている
- `samples:` が 0 より大きい
- `map classes:` が `divider`, `ped_crossing`, `boundary` と対応している

ここが通れば、公式 pkl を baseline として使える状態です。


In [ ]:
from mmcv import Config
from mmcv.utils import import_modules_from_strings
from mmdet3d.datasets import build_dataset
from models.maptr.interface.runtime import import_maptr_plugins

cfg = Config.fromfile(str(CONFIG_PATH))

if cfg.get('custom_imports', None):
    import_modules_from_strings(**cfg.custom_imports)
if cfg.get('plugin', False) and cfg.get('plugin_dir', None):
    plugin_dir = Path(cfg.plugin_dir)
    if not plugin_dir.is_absolute():
        plugin_dir = MAPTR_ROOT / plugin_dir
    import_maptr_plugins(MAPTR_ROOT, plugin_dir)

test_cfg = cfg.data.test.copy()
test_cfg.data_root = str(NUSCENES_ROOT) + os.sep
test_cfg.ann_file = str(ANN_FILE)
test_cfg.pop('map_ann_file', None)
test_cfg.pipeline = cfg.test_pipeline
test_cfg.test_mode = True
print('using split:', SPLIT)
print('using ann_file:', ANN_FILE)

dataset = build_dataset(test_cfg)
print(type(dataset))
print('samples:', len(dataset))
print('map classes:', getattr(dataset, 'MAPCLASSES', None))


## Inspect one official sample: `dataset` -> `raw_info` / `official_info`

`dataset.data_infos[SAMPLE_INDEX]` は、公式 converter が pkl に保存した sample 情報をそのまま取り出したものです。ここでは token、scene、timestamp、map location、camera order を確認します。

`dataset.get_data_info(SAMPLE_INDEX)` は、その raw sample を MapTR dataset class が test pipeline 用に整形した正規入力です。後続 cell で `MapTRSensorInput.to_maptr_info()` と比較するため、ここで key と shape を確認しておきます。


### Output check: one official sample

この cell では、選んだ 1 sample の生情報と、MapTR pipeline に渡す正規入力の形を見ます。

OK の目安:
- `camera order` が `CAM_FRONT`, `CAM_FRONT_RIGHT`, `CAM_FRONT_LEFT`, `CAM_BACK`, `CAM_BACK_LEFT`, `CAM_BACK_RIGHT`
- `image count` が 6
- `can_bus` が `(18,)`
- `lidar2img`, `camera_intrinsics`, `camera2ego` がそれぞれ `(6, 4, 4)`
- `lidar2ego` が `(4, 4)`

この出力が、この後 `types.py` 側で再構成する入力の基準になります。


In [ ]:
raw_info = dataset.data_infos[SAMPLE_INDEX]
official_info = dataset.get_data_info(SAMPLE_INDEX)

print('token:', raw_info['token'])
print('scene_token:', raw_info['scene_token'])
print('frame_idx:', raw_info['frame_idx'])
print('timestamp:', raw_info['timestamp'])
print('map_location:', raw_info['map_location'])
print('camera order:', list(raw_info['cams'].keys()))
print('official keys:', sorted(official_info.keys()))
print('image count:', len(official_info['img_filename']))
print('can_bus:', np.asarray(official_info['can_bus']).shape, np.asarray(official_info['can_bus']).dtype)
print('lidar2img:', np.asarray(official_info['lidar2img']).shape)
print('camera_intrinsics:', np.asarray(official_info['camera_intrinsics']).shape)
print('camera2ego:', np.asarray(official_info['camera2ego']).shape)
print('lidar2ego:', np.asarray(official_info['lidar2ego']).shape)

## Rebuild the same sample: `official_info` -> `rebuilt_input` / `rebuilt_info`

公式 `get_data_info()` から、hdmap-model-bench の共通 MapTR 入力型に詰め直します。これにより `MapTRSensorInput.to_maptr_info()` が公式入力と同じ contract を満たすか確認できます。

### Output check: rebuild into `MapTRSensorInput`

この cell では、公式入力を `types.py` の `MapTRSensorInput` に詰め直します。

OK の目安:
- error が出ない
- `rebuilt keys` に `sample_idx`, `img_filename`, `can_bus`, `lidar2img`, `camera_intrinsics`, `camera2ego`, `lidar2ego` が含まれる

ここはまだ数値比較ではなく、「型に詰め直せるか」の確認です。


In [ ]:
from models.maptr.interface.types import CameraSensor, MapTRSensorInput

cameras = []
for name, image_path, lidar2img, intrinsic, camera2ego in zip(
    raw_info['cams'].keys(),
    official_info['img_filename'],
    official_info['lidar2img'],
    official_info['camera_intrinsics'],
    official_info['camera2ego'],
):
    cameras.append(
        CameraSensor(
            name=name,
            image_path=image_path,
            lidar2img=np.asarray(lidar2img, dtype=np.float32),
            camera_intrinsic=np.asarray(intrinsic, dtype=np.float32),
            camera2ego=np.asarray(camera2ego, dtype=np.float32),
            metadata={'source': 'official_nuscenes'},
        )
    )

rebuilt_input = MapTRSensorInput(
    sample_id=official_info['sample_idx'],
    cameras=tuple(cameras),
    can_bus=np.asarray(official_info['can_bus'], dtype=np.float32),
    timestamp=official_info['timestamp'],
    lidar_path=official_info['pts_filename'],
    lidar2ego=np.asarray(official_info['lidar2ego'], dtype=np.float32),
    metadata={
        'scene_token': official_info['scene_token'],
        'map_location': official_info['map_location'],
        'prev_idx': official_info['prev_idx'],
        'next_idx': official_info['next_idx'],
        'frame_idx': official_info['frame_idx'],
        'ego2global_translation': official_info['ego2global_translation'],
        'ego2global_rotation': official_info['ego2global_rotation'],
        'lidar2ego_translation': official_info['lidar2ego_translation'],
        'lidar2ego_rotation': official_info['lidar2ego_rotation'],
        'sweeps': official_info['sweeps'],
    },
)
rebuilt_info = rebuilt_input.to_maptr_info()
print('rebuilt keys:', sorted(rebuilt_info.keys()))


### Output check: official input vs `types.py` input

この cell が一番重要な contract check です。公式 MapTR 入力 `official_info` と、`MapTRSensorInput.to_maptr_info()` の出力 `rebuilt_info` を比較します。

OK の目安:
- 文字列系 key は `equal=True`
- 行列・配列系 key は `same_shape=True`
- `max_abs=0` または float32 丸め程度のごく小さい値
- 最後に `MapTRSensorInput contract matches the official sample.` が出る

ここが通れば、`types.py` の入力形式は公式 MapTR 入力と同じ contract を満たしています。


In [ ]:
def as_array(value):
    return np.asarray(value)

compare_keys = [
    'sample_idx', 'scene_token', 'pts_filename', 'timestamp', 'img_filename',
    'can_bus', 'lidar2img', 'camera_intrinsics', 'camera2ego', 'lidar2ego',
]

for key in compare_keys:
    left = official_info.get(key)
    right = rebuilt_info.get(key)
    if isinstance(left, (list, tuple, np.ndarray)) and key != 'img_filename':
        left_arr = as_array(left)
        right_arr = as_array(right)
        max_abs = float(np.max(np.abs(left_arr - right_arr))) if left_arr.size else 0.0
        same_shape = left_arr.shape == right_arr.shape
        print(f'{key:20s} shape {left_arr.shape} vs {right_arr.shape} same_shape={same_shape} max_abs={max_abs:.6g}')
    else:
        print(f'{key:20s} equal={left == right}')

assert rebuilt_info['img_filename'] == official_info['img_filename']
assert np.allclose(rebuilt_info['can_bus'], official_info['can_bus'])
assert np.allclose(rebuilt_info['lidar2img'], official_info['lidar2img'])
assert np.allclose(rebuilt_info['camera_intrinsics'], official_info['camera_intrinsics'])
assert np.allclose(rebuilt_info['camera2ego'], official_info['camera2ego'])
assert np.allclose(rebuilt_info['lidar2ego'], official_info['lidar2ego'])
print('MapTRSensorInput contract matches the official sample.')

## Visual image order and projection smoke check

### Output check: camera image order

この grid は camera 順序を人間が見やすい配置に並べたものです。

見るところ:
- 上段が `front_left / front / front_right`
- 下段が `back_left / back / back_right`
- 各画像の向きが camera 名と直感的に合っている

ここで画像が入れ替わって見える場合、camera order や pkl の camera 情報を疑います。


In [ ]:
image_grid_order = [
    ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT'],
    ['CAM_BACK_LEFT', 'CAM_BACK', 'CAM_BACK_RIGHT'],
]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, cam_name in zip(axes.ravel(), sum(image_grid_order, [])):
    cam_info = raw_info['cams'][cam_name]
    image = mmcv.imread(cam_info['data_path'])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    ax.set_title(cam_name)
    ax.axis('off')
plt.tight_layout()


### How to read the projection smoke check

次の cell は、lidar/ego 座標の簡単な probe points を各 camera 画像へ `lidar2img` で投影し、画像内に入った点数と pixel 座標を表示します。

- `visible probes` が多い camera は、その probe points がその camera の視野内にあるという意味です。
- `CAM_FRONT` では前方の点 `[5, 0, 0]`, `[10, 0, 0]`, `[15, 0, 0]` が見えるのが自然です。
- `CAM_FRONT_LEFT` / `CAM_FRONT_RIGHT` では左右の probe がどちらに出るかを見て、左右反転が疑わしくないか確認します。
- `depth` が負または極端に小さい点は、その camera の後方または投影不能な点です。
- ここで見るのは精密な精度ではなく、camera order と `lidar2img` の向きが大きく破綻していないかです。


In [ ]:
# Project a few simple lidar-frame points into each camera.
probe_points = np.asarray([
    [5, 0, 0, 1], [10, 0, 0, 1], [15, 0, 0, 1],
    [10, 5, 0, 1], [10, -5, 0, 1],
], dtype=np.float32)

for cam_name, image_path, lidar2img in zip(raw_info['cams'].keys(), official_info['img_filename'], official_info['lidar2img']):
    image = mmcv.imread(image_path)
    height, width = image.shape[:2]
    projected = np.asarray(lidar2img) @ probe_points.T
    depth = projected[2]
    uv = (projected[:2] / np.maximum(depth, 1e-6)).T
    visible = (depth > 0) & (uv[:, 0] >= 0) & (uv[:, 0] < width) & (uv[:, 1] >= 0) & (uv[:, 1] < height)
    print(f'{cam_name:16s} visible probes: {int(visible.sum())}/{len(probe_points)} uv={np.round(uv, 1).tolist()} depth={np.round(depth, 2).tolist()}')

## Inspect pipeline output: `dataset` -> `example`

### Output check: pipeline output

この cell では、公式 MapTR test pipeline を通した後の tensor / metadata 構造を見ます。

OK の目安:
- `prepared keys` 相当として `img`, `img_metas` がある
- `img` が 6 camera 分の tensor を含む
- `meta sample_idx` が上で確認した sample token と一致する
- `meta keys` に `lidar2img`, `can_bus`, `camera_intrinsics`, `camera2ego`, `lidar2ego` が含まれる

ここが通れば、公式入力は model に渡せる形まで整形できています。


In [ ]:
example = dataset.prepare_test_data(SAMPLE_INDEX)

from models.maptr.interface.predictor import MapTRPredictor

def unwrap(value):
    if hasattr(value, '_data'):
        return unwrap(value._data)
    if hasattr(value, 'data') and value.__class__.__name__ == 'DataContainer':
        return unwrap(value.data)
    if isinstance(value, dict):
        return {k: unwrap(v) for k, v in value.items()}
    if isinstance(value, tuple):
        return tuple(unwrap(v) for v in value)
    if isinstance(value, list):
        return [unwrap(v) for v in value]
    return value

def describe(value):
    if hasattr(value, 'shape'):
        return f'{type(value).__name__} shape={tuple(value.shape)} dtype={getattr(value, "dtype", None)}'
    if isinstance(value, list):
        return f'list len={len(value)} first={describe(value[0]) if value else "empty"}'
    if isinstance(value, dict):
        return f'dict keys={list(value.keys())[:16]}'
    return repr(value)[:160]

unwrapped = unwrap(example)
for key, value in unwrapped.items():
    print(f'{key:12s} -> {describe(value)}')

metas = unwrapped['img_metas']
while isinstance(metas, (list, tuple)):
    metas = metas[0]
print('meta sample_idx:', metas.get('sample_idx'))
print('meta keys:', sorted(metas.keys()))

## Run inference: `example` -> `raw_output` / `prediction`

### Output check: predictor setup

この cell では、config / checkpoint から MapTR model を構築します。

OK の目安:
- checkpoint load error が出ない
- `device: cuda:0` のように CUDA device が表示される
- `map classes` が expected class と一致する

ここで落ちる場合は、入力データではなく MapTR 実行環境、checkpoint、runtime patch 側の問題である可能性が高いです。


In [ ]:
predictor = MapTRPredictor(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    maptr_root=MAPTR_ROOT,
    device=DEVICE,
    score_threshold=SCORE_THRESHOLD,
)
print('device:', predictor.device)
print('map classes:', predictor.map_classes)

### Output check: raw prediction

この cell では、公式 dataset 経由の `example` を使って MapTR 推論を実行し、raw output と整形済み prediction を確認します。

OK の目安:
- `pts_bbox keys` に `scores_3d`, `labels_3d`, `pts_3d` が含まれる
- `scores_3d`, `labels_3d`, `pts_3d` の件数が対応している
- `vectors after threshold` が表示される
- `by class` で class ごとの件数が見える

この cell は `rebuilt_input` ではなく、公式 dataset が作った `example` で baseline 推論を確認しています。


In [ ]:
raw_output = predictor.predict_raw(example)
prediction = predictor.format_output(
    raw_output,
    sample_id=official_info['sample_idx'],
    score_threshold=SCORE_THRESHOLD,
    raw_output_field=False,
)

detection = predictor._extract_pts_bbox(raw_output)
print('raw type:', type(raw_output))
print('pts_bbox keys:', list(detection.keys()))
for key, value in detection.items():
    print(f'{key:16s} -> {describe(value)}')
print('vectors after threshold:', len(prediction.vectors))
print('by class:', {k: len(v) for k, v in prediction.by_class().items()})

### Output check: ground-truth vector annotations

公式 converter が作った正解データは `raw_info['annotation']` に入っています。ここでは MapTR が出力する class (`divider`, `ped_crossing`, `boundary`) だけを取り出して `gt_vectors` にそろえます。

見るところ:
- `gt vectors by class` が 0 ばかりではない
- `ignored gt classes` に `centerline` が出ることがあるが、これは MapTR の標準 3 class には含めないので問題ありません
- 各 vector の点列はすでに ego/lidar 周辺の local BEV 座標です


In [ ]:
gt_annotations = raw_info.get('annotation', {})
gt_vectors = []

for class_id, class_name in enumerate(predictor.map_classes):
    for points in gt_annotations.get(class_name, []):
        pts = np.asarray(points, dtype=float)
        if pts.ndim != 2 or pts.shape[0] == 0 or pts.shape[1] < 2:
            continue
        gt_vectors.append({
            'class_id': class_id,
            'class_name': class_name,
            'points': pts[:, :2],
        })

print('gt annotation keys:', sorted(gt_annotations.keys()))
print('gt vectors by class:', {cls: len(gt_annotations.get(cls, [])) for cls in predictor.map_classes})
print('ignored gt classes:', sorted(set(gt_annotations) - set(predictor.map_classes)))
print('gt vectors used for visualization:', len(gt_vectors))
if gt_vectors:
    lengths = [len(v['points']) for v in gt_vectors]
    print('points per gt vector: min/median/max =', min(lengths), int(np.median(lengths)), max(lengths))


### Output check: BEV ground truth and prediction visualization

1つ目の図は公式変換済み pkl に含まれる正解 vector map です。2つ目の図は同じ座標系に正解と推論を重ねています。

見るところ:
- GT-only 図で、正解線が `point_cloud_range` の範囲内に自然に入っている
- overlay 図で、点線の GT と実線の prediction が大きく座標ずれしていない
- 左右反転、前後反転、全体のスケール違い、原点から極端に離れた出力がない

線種:
- 点線: official GT (`raw_info['annotation']`)
- 実線: MapTR prediction (`prediction.vectors`)


In [ ]:
class_colors = {'divider': '#1f77b4', 'ped_crossing': '#2ca02c', 'boundary': '#d62728'}
pc_range = np.asarray(cfg.point_cloud_range, dtype=float)
x_min, y_min, _, x_max, y_max, _ = pc_range

def setup_bev_axis(ax, title):
    ax.set_title(title)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('x forward [m]')
    ax.set_ylabel('y left [m]')
    ax.grid(True, alpha=0.25)
    ax.set_aspect('equal', adjustable='box')
    ax.scatter([0], [0], marker='x', color='black', label='ego')

def dedupe_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(
        unique.values(),
        unique.keys(),
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        frameon=True,
    )

fig, ax = plt.subplots(figsize=(8.5, 10))
setup_bev_axis(ax, f'Official GT {official_info["sample_idx"][:8]}')
for gt in gt_vectors:
    pts = gt['points']
    ax.plot(
        pts[:, 0], pts[:, 1],
        color=class_colors.get(gt['class_name'], '#555555'),
        linewidth=2.0,
        linestyle='--',
        alpha=0.85,
        label=f'GT {gt["class_name"]}',
    )
dedupe_legend(ax)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 10))
setup_bev_axis(ax, f'GT vs MapTR prediction {official_info["sample_idx"][:8]} threshold={SCORE_THRESHOLD}')
for gt in gt_vectors:
    pts = gt['points']
    ax.plot(
        pts[:, 0], pts[:, 1],
        color=class_colors.get(gt['class_name'], '#555555'),
        linewidth=2.2,
        linestyle='--',
        alpha=0.65,
        label=f'GT {gt["class_name"]}',
    )
for vector in prediction.vectors:
    pts = np.asarray(vector.points, dtype=float)
    if len(pts) == 0:
        continue
    ax.plot(
        pts[:, 0], pts[:, 1],
        color=class_colors.get(vector.class_name, '#555555'),
        linewidth=1.5,
        linestyle='-',
        alpha=max(0.35, min(1.0, vector.score)),
        label=f'Pred {vector.class_name}',
    )
dedupe_legend(ax)
fig.tight_layout()
plt.show()


### Output check: saved normalized prediction

この cell では、`MapTRPrediction.to_dict()` の結果を JSON に保存します。

OK の目安:
- `outputs/maptr/nusc/<sample_token>.json` の path が表示される
- JSON の `sample_token` がこの notebook で見ている sample と一致する
- `vectors` に `pts`, `pts_num`, `cls_name`, `type`, `confidence_level` が入る

この JSON は、後で evaluation 側や adapter 経由の出力と比較するための保存結果です。


In [ ]:
import json

output_dir = WORKSPACE_ROOT / 'outputs' / 'maptr' / 'nusc'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{official_info["sample_idx"]}.json'
with output_path.open('w', encoding='utf-8') as f:
    json.dump(prediction.to_dict(), f, indent=2)
print(output_path)
